In [1]:
%%time
%matplotlib inline
import rioxarray
import numpy as np
import importlib 
import matplotlib.pyplot as plt
import geopandas as gpd
import os
import glob

CPU times: total: 984 ms
Wall time: 1.5 s


In [2]:
# Code này dùng để lọc ra các điểm VH và NDVI không bị NaN trong suốt 13 tháng
# THis code extracting non-nan point of VH and NDVI over 13 months

In [3]:
# Đường dẫn thư mục chứa các file mask
input_folder = "./input/local"
mask_folder = "./input/mask"
output_folder = "./input/non_nan"

In [4]:
ndvi_data = rioxarray.open_rasterio(os.path.join(input_folder, "STNDVI_202223.tif"))
vh_data = rioxarray.open_rasterio(os.path.join(input_folder, "STVH_202223.tif"))

In [ ]:
# Lấy tất cả file .tif trừ NDVI và VH
tif_files = glob.glob(os.path.join(mask_folder, "*.tif"))
# tif_files = [f for f in tif_files if "STNDVI" not in f and "STVH" not in f]

for tif_file in tif_files:
    mask_data = rioxarray.open_rasterio(tif_file)
    file_stem = os.path.splitext(os.path.basename(tif_file))[0]  # tên file không có đuôi

    # Áp dụng mask
    mask = mask_data != 255
    data_stacked = mask_data.stack(points=("y", "x"))
    filtered = data_stacked.where(data_stacked != 255, drop=True)

    x_indices = filtered.coords["x"].values
    y_indices = filtered.coords["y"].values
    xy_valid = np.column_stack((x_indices, y_indices))

    ndvi_list = []
    vh_list = []
    for index in xy_valid:
        data = ndvi_data.sel(x=index[0], y=index[1], method='nearest').values
        vh = vh_data.sel(x=index[0], y=index[1], method='nearest').values
        if not np.any(np.isnan(data)):
            ndvi_list.append(data)
            vh_list.append(vh)

    # Lưu riêng từng file NDVI/VH không chứa NaN
    ndvi_outfile = os.path.join(output_folder, f"{file_stem}_nonan_ndvi.npy")
    vh_outfile = os.path.join(output_folder, f"{file_stem}_nonan_vh.npy")
    
    np.save(ndvi_outfile, ndvi_list)
    np.save(vh_outfile, vh_list)

    print(f"Đã lưu: {ndvi_outfile}, {vh_outfile} - Số điểm: {len(ndvi_list)}")

Đã lưu: ./input/non_nan\ST_TS_nonan_ndvi.npy, ./input/non_nan\ST_TS_nonan_vh.npy - Số điểm: 2618
Đã lưu: ./input/non_nan\ST_CHN_nonan_ndvi.npy, ./input/non_nan\ST_CHN_nonan_vh.npy - Số điểm: 26
